In [1]:
%matplotlib qt

import numpy as np
import hyperspy.api as hs
from hyperspy.signals import Signal2D
import gc

In [2]:
#Load the data
Data = hs.load(r"C:\User\Sample A\4dstem3\2.hspy", reader="hspy")
Data.plot()

In [ ]:
#Sometimes significant bleedthrough can ruin individual patterns on the navigational axes
#This code attemps to find and replace these bleedthrough events with the mean of the pattern
#It should be used mindfully, as STEM images containing large contrasts in signal, may have those high intensity areas replaced.

TotalIntensity = Data.data.sum(axis=(-2, -1))

Median = np.median(TotalIntensity)
MedianAbsDeviation = np.median(np.abs(TotalIntensity - Median))

Threshold = median + 13 * MedianAbsDeviation
Mask = TotalIntensity > Threshold

Mean = Data.data[~Mask].mean(axis=0)

# Replace bad pixels
Data.data[Mask] = Mean

gc.collect()

In [9]:
Data.set_signal_type('electron_diffraction')  # assign electron diffraction so pyxem can work
gc.collect()
Data.data = Data.data.astype('float32')
Data.data *= 1 / Data.data.max()

In [ ]:
#Rebin if necessary
Data = dp.rebin(scale=(1, 1, 2, 2))

In [11]:
#Select the region, widget working!
Region = hs.roi.RectangularROI(left=-100, top=-10, right=300, bottom=300)
Data.plot(cmap='inferno', vmax=0.015)
Region.add_widget(Data)

In [ ]:
#Assign the region with coordinates if widget is not working 
Region = hs.roi.RectangularROI(left=10, top=10, right=50, bottom=20)

In [13]:
#Bright field, dark dield and integrated spot imaging
Cropped_Data = Region(Data)
Integrated = Cropped_Data.sum()
Integrated.plot(cmap = 'inferno_r', vmax = 100)

L = Data.sum(axis=(-1, -2)) 
#Bright field image:
#Signal2D((np.array(L))).plot(cmap = 'inferno')

#Dark field image:
Signal2D((100000 - np.array(L))).plot(cmap = 'inferno')
